**Kaushik Budur**

MyMav ID: 1002224112


Code to compute the 2-way orthologs between T. castaneum and A. glabripennis


In [8]:
from google.colab import files
uploaded = files.upload()  # Pick all 6 files at once

# STEP 2: Helper functions
import pandas as pd
from pathlib import Path

def get_best_hits(file_path: str):
    """
    Given a BLAST tab file (qseqid, sseqid, evalue),
    return a dictionary: query -> best subject (lowest e-value).
    """
    df = pd.read_csv(file_path, sep="\t", header=None, engine="python")
    df = df.iloc[:, :3]
    df.columns = ["query", "subject", "evalue"]
    df["evalue"] = pd.to_numeric(df["evalue"], errors="coerce")
    df = df.sort_values(["query", "evalue"], ascending=[True, True])
    best_hits = df.drop_duplicates(subset="query", keep="first")
    return dict(zip(best_hits["query"], best_hits["subject"]))

def reciprocal_hits(file1: str, file2: str):
    """
    Compute reciprocal best hits between two BLAST outputs:
      - file1: species A → species B
      - file2: species B → species A
    Returns a set of (geneA, geneB) pairs.
    """
    best1 = get_best_hits(file1)
    best2 = get_best_hits(file2)
    return {(a, b) for a, b in best1.items() if best2.get(b) == a}

# STEP 3: Compute RBH for each species pair
rbh_Tcas_Ldec = reciprocal_hits("Tcas_query_v_Ldec_subject.txt",
                                "Ldec_query_v_Tcas_subject.txt")

rbh_Agla_Ldec = reciprocal_hits("Agla_query_v_Ldec_subject.txt",
                                "Ldec_query_v_Agla_subject.txt")

rbh_Agla_Tcas = reciprocal_hits("Agla_query_v_Tcas_subject.txt",
                                "Tcas_query_v_Agla_subject.txt")

# STEP 4: Display results
print("Reciprocal Best-Hit (RBH) ortholog pair counts:")
print(f"T. castaneum  ↔  L. decemlineata : {len(rbh_Tcas_Ldec)}")
print(f"A. glabripennis ↔ L. decemlineata : {len(rbh_Agla_Ldec)}")
print(f"A. glabripennis ↔ T. castaneum    : {len(rbh_Agla_Tcas)}")

# STEP 5: (Optional) Save RBH pairs as CSV files
pd.DataFrame(sorted(rbh_Tcas_Ldec), columns=["Tcas_gene","Ldec_gene"]) \
  .to_csv("RBH_Tcas_Ldec.csv", index=False)

pd.DataFrame(sorted(rbh_Agla_Ldec), columns=["Agla_gene","Ldec_gene"]) \
  .to_csv("RBH_Agla_Ldec.csv", index=False)

pd.DataFrame(sorted(rbh_Agla_Tcas), columns=["Agla_gene","Tcas_gene"]) \
  .to_csv("RBH_Agla_Tcas.csv", index=False)

print("\n✅ CSV files saved in Colab working directory.")


Saving Tcas_query_v_Ldec_subject.txt to Tcas_query_v_Ldec_subject.txt
Saving Ldec_query_v_Agla_subject.txt to Ldec_query_v_Agla_subject.txt
Saving Tcas_query_v_Agla_subject.txt to Tcas_query_v_Agla_subject.txt
Saving Ldec_query_v_Tcas_subject.txt to Ldec_query_v_Tcas_subject.txt
Saving Agla_query_v_Tcas_subject.txt to Agla_query_v_Tcas_subject.txt
Saving Agla_query_v_Ldec_subject.txt to Agla_query_v_Ldec_subject.txt
Reciprocal Best-Hit (RBH) ortholog pair counts:
T. castaneum  ↔  L. decemlineata : 9509
A. glabripennis ↔ L. decemlineata : 9921
A. glabripennis ↔ T. castaneum    : 10250

✅ CSV files saved in Colab working directory.


How many 2-way orthologous pairs are there between A. glabripennis and T. castaneum?

10,250 Pairs

In [9]:
def parse_blast_file(filename):
    """Parse BLAST tabular output to get best hit per query (first hit assumed best)."""
    best_hits = {}
    with open(filename) as f:
        for line in f:
            if line.strip() == '' or line.startswith('#'):
                continue
            parts = line.strip().split()
            query, subject = parts[0], parts[1]
            if query not in best_hits:
                best_hits[query] = subject
    return best_hits

# File pairs mapped to output filenames
file_pairs = [
    ("Agla_query_v_Tcas_subject.txt", "Tcas_query_v_Agla_subject.txt", "reciprocal_hits_Agla_Tcas.txt"),
    ("Ldec_query_v_Agla_subject.txt", "Agla_query_v_Ldec_subject.txt", "reciprocal_hits_Ldec_Agla.txt"),
    ("Tcas_query_v_Ldec_subject.txt", "Ldec_query_v_Tcas_subject.txt", "reciprocal_hits_Tcas_Ldec.txt"),
]

for file_a, file_b, out_filename in file_pairs:
    hits_a = parse_blast_file(file_a)
    hits_b = parse_blast_file(file_b)
    reciprocal_hits = []
    for query, hit in hits_a.items():
        if hit in hits_b and hits_b[hit] == query:
            reciprocal_hits.append((query, hit))

    with open(out_filename, 'w') as outfile:
        outfile.write(f'Reciprocal best hits for {file_a} and {file_b}:\n')
        outfile.write(f'Total pairs: {len(reciprocal_hits)}\n')
        for pair in reciprocal_hits:
            outfile.write(f'{pair[0]}\t{pair[1]}\n')

    print(f'Saved {len(reciprocal_hits)} reciprocal hits to {out_filename}')


Saved 10250 reciprocal hits to reciprocal_hits_Agla_Tcas.txt
Saved 9921 reciprocal hits to reciprocal_hits_Ldec_Agla.txt
Saved 9509 reciprocal hits to reciprocal_hits_Tcas_Ldec.txt


In [11]:
def parse_blast_file(filename):
    """Parse BLAST tabular output to get best hit per query (first hit assumed best)."""
    best_hits = {}
    with open(filename) as f:
        for line in f:
            if line.strip() == '' or line.startswith('#'):
                continue
            parts = line.strip().split()
            query, subject = parts[0], parts[1]
            if query not in best_hits:  # only keep the first (best) hit
                best_hits[query] = subject
    return best_hits

# File pair: T. castaneum ↔ A. glabripennis
file_a = "Agla_query_v_Tcas_subject.txt"   # A. glabripennis as query
file_b = "Tcas_query_v_Agla_subject.txt"   # T. castaneum as query
out_filename = "reciprocal_hits_Agla_Tcas.txt"

# Parse best hits
hits_a = parse_blast_file(file_a)
hits_b = parse_blast_file(file_b)

# Identify reciprocal best hits (2-way orthologs)
reciprocal_hits = []
for query, hit in hits_a.items():
    if hit in hits_b and hits_b[hit] == query:
        reciprocal_hits.append((query, hit))

# Save results
with open(out_filename, 'w') as outfile:
    outfile.write(f'Reciprocal best hits between A. glabripennis and T. castaneum:\n')
    outfile.write(f'Total pairs: {len(reciprocal_hits)}\n')
    for pair in reciprocal_hits:
        outfile.write(f'{pair[0]}\t{pair[1]}\n')

print(f'Saved {len(reciprocal_hits)} reciprocal hits to {out_filename}  2-way orthologs between T. castaneum and A. glabripennis')


Saved 10250 reciprocal hits to reciprocal_hits_Agla_Tcas.txt  2-way orthologs between T. castaneum and A. glabripennis
